In [1]:
import sys, os
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [2]:
from src.multimodal import MultimodalDataset

ds = MultimodalDataset.load('dataset/preprocessed_dataset.npz', batch_size=16)


In [3]:
import tensorflow as tf
from src.models import EarEncoder, HRTFEncoder, ContrastiveModel
import numpy as np

enc_ear  = EarEncoder(embedding_dim=128)
enc_hrtf = HRTFEncoder(n_sh=121, n_freqs=129, embedding_dim=128)
model    = ContrastiveModel(enc_ear, enc_hrtf, temperature=0.07)

# Build le modèle avec un batch factice
dummy = {
    'ear_left':  np.zeros((1, 224, 224, 3), dtype='float32'),
    'ear_right': np.zeros((1, 224, 224, 3), dtype='float32'),
    'hrtf':      np.zeros((1, 121, 129, 2), dtype='float32'),
}

model(dummy, training=False)   # ← build le graphe

# Chemin vers les poids sauvegardés — adapte selon ton expérience
weights_path = 'checkpoints/exp_temp007_bs8_2d83030a/phase1_best.weights.h5'
model.load_weights(weights_path)
print(f'Poids chargés depuis : {weights_path}')


Poids chargés depuis : checkpoints/exp_temp007_bs8_2d83030a/phase1_best.weights.h5


In [4]:
from src.evaluation import EmbeddingSpace, EmbeddingVisualizer, evaluate

space = EmbeddingSpace.compute(model, ds)
space.save('checkpoints/embeddings.npz')
print(space)


  train — 57 sujets encodés
  val   — 11 sujets encodés
  test  — 12 sujets encodés
  EmbeddingSpace sauvegardé → checkpoints/embeddings.npz
EmbeddingSpace(N=80, D=128, splits={'train': 57, 'val': 11, 'test': 12})


In [5]:
evaluate(space, split='test')


  Évaluation — split : test  (12 sujets)
  ────────────────────────────────────────
  Recall@1  : 0.167   (2/12 sujets)
  Recall@3  : 0.333
  Recall@5  : 0.417
  MRR       : 0.334


{'recall@1': 0.16666666666666666,
 'recall@3': 0.3333333333333333,
 'recall@5': 0.4166666666666667,
 'mrr': 0.33356481481481476}

In [6]:
#C'est la visualisation centrale qui répond directement à la question :
#  "Le modèle aligne-t-il correctement les deux modalités ?"
viz = EmbeddingVisualizer(space)
viz.plot_alignment(split='test')


  UMAP en cours sur 24 vecteurs...


c:\son_spatialisation\.venv_ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\son_spatialisation\.venv_ml\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [10]:
viz.plot_alignment(split='train')

  UMAP en cours sur 114 vecteurs...


c:\son_spatialisation\.venv_ml\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [7]:
#Elle répond à une question différente : "L'espace latent a-t-il une 
#structure géographique — les sujets similaires sont-ils proches ?"
#Et elle permet de comparer les deux espaces séparément :
#est-ce que la carte z_ear ressemble à la carte z_hrtf ?
viz.plot_subjects(modality='ear', split='test')

  UMAP en cours sur 12 vecteurs z_ear...


c:\son_spatialisation\.venv_ml\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\son_spatialisation\.venv_ml\Lib\site-packages\umap\umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


In [8]:
viz.plot_subjects(modality='hrtf', split='test')

  UMAP en cours sur 12 vecteurs z_hrtf...


c:\son_spatialisation\.venv_ml\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\son_spatialisation\.venv_ml\Lib\site-packages\umap\umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


In [9]:
#Elle valide que les augmentations sont cohérentes
viz.plot_augmentations('H10')

  UMAP sur 4 variantes de H10...


c:\son_spatialisation\.venv_ml\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
